# Example code
This notebook runs the recommendation took for sample pair repository

Import os and set the directory to the parent directory ```MSCEDD/```

In [ ]:
import os  
os.chdir('./..')

Import necessary libraries

In [ ]:
import sys
import pickle
import pandas as pd
from Examples.PaReco import PaReco
import Methods.common as common

try:
    import argparse
    import magic
except ImportError as err:
    print(err)
    sys.exit(-1)


Add GitHub tokens

GitHub tokens can be added in two ways (choose one):
    
    1. Read from file from tokens.txt
    
    2. Manually add to the token_list in the notebook. 
    token_list = ['token1', 'token2', 'token3', ..., 'tokenx']

Keep in mind that GitHub tokens are automatically reverted when commited to GitHub. Before commiting always make sure to remove all tokens from the notebook ifyou add it in the notebook.

In [ ]:
"""
    Read from list
"""
# token_file = 'tokens.txt'
# token_file = 'tokens.txt'

# token_list = list()
# with open(token_file) as f:
#     lines = f.readlines()
#     for line in lines[0].split(','):
#         token_list.append(line)

"""
    Manually add in notebook
"""
token_list = [

]

lenTokens = len(token_list)
print(lenTokens)

Create a magic cookie. This is needed during classification

In [ ]:
common.verbose = False

try:
    common.magic_cookie = magic.open(magic.MAGIC_MIME)
    common.magic_cookie.load()
except AttributeError:
    common.magic_cookie = magic.Magic(mime=True, uncompress=True)
common.verbose_print('[-] initialized magic cookie\n')

Create the direcotires Repos_files and Repos_results that will be used to store data while running.

Run this only once to create the directories and after comment it out.

In [ ]:
os.mkdir(os.path.join(os.getcwd(), 'Examples/Repos_files'))
os.mkdir(os.path.join(os.getcwd(), 'Examples/Repos_results'))

### Running multiple repository pairs automatically in six-month intervals

Add matching repositories to `variant1_list` and `variant2_list`. Items at the same Python list index are always processed together:

- `variant1_list[0]` runs with `variant2_list[0]`.
- `variant1_list[1]` runs with `variant2_list[1]`.
- Both lists must contain the same number of entries.
- Each pair retrieves its own divergence and cutoff dates.
- Every start timestamp uses `T00:00:00Z`.
- Every end timestamp uses `T23:59:59Z`.
- The next interval starts one day after the previous interval ends.
- The final interval for each pair is capped at the earlier of that pair's cutoff date and the global hard limit `2026-09-03T23:59:59Z`.
- The data ID starts at `42` and increments globally across all pairs and intervals.


In [ ]:
starting_data_number = 1

# Repositories at the same list index are paired together.
variant2_list = [
    "tduehr/omniauth-cas3",
    "puppetlabs/dependency_checker",
    "tagliala/coveralls-ruby-reborn",
    "PowerShell/openssh-portable",
    "open-quantum-safe/openssl",
    "apimatic/jsonmapper",
    "ploxiln/paramiko-ng",
    "Cosgy-Dev/JMusicBot-JP",
    "langerhans/dogecoinj-new",
    "wolfemm/email_assessor",
    "pschmitt/roombapy",
    "ashmaroli/jekyll-manager",
    "huacnlee/rails-settings-cached",
    "mozilla/redash",
    "HewlettPackard/python-ilorest-library",
    "yourlabs/django-cities-light",
    "Electron-Cash/Electron-Cash",
    "gmazza/tightblog",
    "cotiga/voyager",
    "cpplint/cpplint",
    "scylladb/python-driver",
    "phax/ParserGeneratorCC",
    "E3SM-Project/scorpio",
    "uploadcare/pillow-simd",
    "Unity-Technologies/curl",
    "twitter-forks/presto",
    "Willena/sqlite-jdbc-crypt",
    "espressif/esp-nimble",
    "ComputeCanada/easybuild-easyconfigs",
    "exoeconomy/EXOS-Electrum",
    "Groestlcoin/electrum-grs",
    "nakagami/CyMySQL",
    "mcpt/wlmoj",
    "ufal/clarin-dspace",
    "lightstep/ls-trace-rb", # Gone
    "appsembler/configuration",
    "Stackdriver/collectd",
    "zsa/qmk_firmware",
    "JetBrains/azure-tools-for-intellij",
    "germ/qmk_firmware",
    "BlueBrain/spack",
    "linkedin/kafka",
]

variant1_list = [
    "dlindahl/omniauth-cas",
    "ekohl/metadata_json_deps",
    "lemurheavy/coveralls-ruby",
    "openssh/openssh-portable",
    "openssl/openssl",
    "cweiske/jsonmapper",
    "paramiko/paramiko",
    "jagrosh/MusicBot",
    "bitcoinj/bitcoinj",
    "micke/valid_email2",
    "NickWaterton/Roomba980-Python",
    "jekyll/jekyll-admin",
    "ledermann/rails-settings",
    "getredash/redash",
    "DMTF/python-redfish-library",
    "coderholic/django-cities",
    "spesmilo/electrum",
    "apache/roller",
    "the-control-group/voyager",
    "google/styleguide",
    "datastax/python-driver",
    "javacc/javacc",
    "NCAR/ParallelIO",
    "python-pillow/Pillow",
    "curl/curl",
    "prestodb/presto",
    "xerial/sqlite-jdbc",
    "apache/mynewt-nimble",
    "easybuilders/easybuild-easyconfigs",
    "spesmilo/electrum",
    "spesmilo/electrum",
    "PyMySQL/PyMySQL",
    "DMOJ/online-judge",
    "DSpace/DSpace",
    "DataDog/dd-trace-rb",
    "edx/configuration",
    "collectd/collectd",
    "qmk/qmk_firmware",
    "microsoft/azure-tools-for-java",
    "qmk/qmk_firmware",
    "spack/spack",
    "apache/kafka",
]

# variant1_list = ['apache/kafka']
# variant2_list = ['linkedin/kafka']

if not variant1_list:
    raise ValueError('At least one variant pair is required.')

if len(variant1_list) != len(variant2_list):
    raise ValueError(
        'variant1_list and variant2_list must have the same number of entries. '
        f'Got {len(variant1_list)} and {len(variant2_list)}.'
    )

variant_pairs = list(zip(variant1_list, variant2_list))
print(f'Configured {len(variant_pairs)} variant pair(s).')


Create contiguous six-month date windows. Calendar months are used instead of a fixed number of days, so the day-of-month behavior matches the requested PR dataset names. The global hard limit is `2026-09-03T23:59:59Z`, so no generated window can extend beyond September 3, 2026.


In [ ]:
from calendar import monthrange
from datetime import date, datetime, timedelta, timezone


# Absolute upper bound for every variant date window/check.
HARD_VARIANT_END = datetime(2026, 9, 3, 23, 59, 59, tzinfo=timezone.utc)
HARD_VARIANT_END_TIMESTAMP = '2026-09-03T23:59:59Z'


def as_datetime(value):
    """Convert a datetime or ISO-8601 string to a timezone-aware UTC datetime."""
    if isinstance(value, datetime):
        parsed = value
    elif isinstance(value, date):
        parsed = datetime(value.year, value.month, value.day, tzinfo=timezone.utc)
    else:
        text = str(value).strip()
        if text.endswith('Z'):
            text = text[:-1] + '+00:00'
        parsed = datetime.fromisoformat(text)

    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=timezone.utc)

    return parsed.astimezone(timezone.utc)


def as_date(value):
    """Convert a date, datetime, or ISO-8601 string to a date."""
    if isinstance(value, datetime):
        return as_datetime(value).date()
    if isinstance(value, date):
        return value

    return as_datetime(value).date()


def add_calendar_months(value, months):
    """Add calendar months while safely clamping the day for shorter months."""
    month_index = value.month - 1 + months
    year = value.year + month_index // 12
    month = month_index % 12 + 1
    day = min(value.day, monthrange(year, month)[1])
    return value.replace(year=year, month=month, day=day)


def build_six_month_intervals(diverge_date, cut_off_date, first_data_number=42):
    """Build inclusive, contiguous six-month windows with incrementing IDs."""
    current_start = as_date(diverge_date)
    requested_cutoff = as_date(cut_off_date)
    final_cutoff = min(requested_cutoff, HARD_VARIANT_END.date())

    if current_start > final_cutoff:
        raise ValueError(
            'The divergence date must not be later than the effective cutoff date '
            f'({final_cutoff:%Y-%m-%d}).'
        )

    intervals = []
    data_number = int(first_data_number)

    while current_start <= final_cutoff:
        current_end = min(
            add_calendar_months(current_start, 6) - timedelta(days=1),
            final_cutoff
        )

        intervals.append({
            'data_number': str(data_number),
            'label': f'{current_start:%Y_%m_%d}_{current_end:%Y_%m_%d}',
            'start': f'{current_start:%Y-%m-%d}T00:00:00Z',
            'end': f'{current_end:%Y-%m-%d}T23:59:59Z',
        })

        current_start = current_end + timedelta(days=1)
        data_number += 1

    return intervals


Retrieve the repository dates for every index-matched pair, then generate one combined interval plan. If date retrieval fails for one pair, the failure is recorded and the remaining pairs are still prepared.


In [ ]:
intervals = []
pair_setup_summary = []
date_probes = {}
next_data_number = int(starting_data_number)

for variant_index, (variant1, variant2) in enumerate(variant_pairs):
    print('=' * 80)
    print(f'Preparing variant index {variant_index}: {variant1} -> {variant2}')

    # This object is used only to retrieve this pair's divergence and cutoff dates.
    date_probe_data = (
        str(next_data_number),
        variant1,
        variant2,
        '',
        '',
        token_list,
    )
    date_probe = PaReco(date_probe_data)
    date_probes[variant_index] = date_probe

    try:
        date_probe.getDates()
        pair_intervals = build_six_month_intervals(
            date_probe.diverge_date,
            date_probe.cut_off_date,
            next_data_number,
        )

        for window in pair_intervals:
            window.update({
                'variant_index': variant_index,
                # Canonical names, as GACPD uses for renamed/transferred repos.
                'variant1': date_probe.variant1,
                'variant2': date_probe.variant2,
            })

        intervals.extend(pair_intervals)
        interval_count = len(pair_intervals)

        if pair_intervals:
            next_data_number = int(pair_intervals[-1]['data_number']) + 1

        setup_status = 'Ready'

    except Exception as exc:
        interval_count = 0
        setup_status = f'Failed: {type(exc).__name__}: {exc}'
        print(setup_status)

    pair_setup_summary.append({
        'variant_index': variant_index,
        'variant1': variant1,
        'variant2': variant2,
        'interval_count': interval_count,
        'status': setup_status,
    })

interval_plan_columns = [
    'data_number',
    'variant_index',
    'variant1',
    'variant2',
    'label',
    'start',
    'end',
]
interval_plan = pd.DataFrame(intervals, columns=interval_plan_columns)
pair_setup_summary_df = pd.DataFrame(pair_setup_summary)

# Save the plan next to the result pickles. Table_Image_Generation uses it to
# map every result to its pair and dates exactly. One file per run direction,
# named after the first pair's variant1 (e.g. interval_plan_dlindahl_omniauth-cas.csv).
interval_plan_path = os.path.join(
    'Examples', 'Repos_results',
    'interval_plan_' + variant_pairs[0][0].replace('/', '_') + '.csv',
)
interval_plan.to_csv(interval_plan_path, index=False)
print(f'Saved interval plan: {interval_plan_path}')

Review the pair setup results and generated data IDs before running classification. The `variant_index` column shows exactly which entries from the two lists were paired.


In [ ]:
display(pair_setup_summary_df)
display(interval_plan)


Run extraction and classification for every generated interval from every repository pair. Each interval receives a unique, globally incremented data ID. A failure in one interval is recorded without preventing later intervals or pairs from running.


In [ ]:
interval_runs = {}
run_summary = []

for window in intervals:
    data_number = window['data_number']
    variant_index = window['variant_index']
    variant1 = window['variant1']
    variant2 = window['variant2']

    print('=' * 80)
    print(f'Variant index {variant_index}: {variant1} -> {variant2}')
    print(f"Data {data_number}: {window['label']}")
    print(f"Range: {window['start']} -> {window['end']}")

    data = (
        data_number,
        variant1,
        variant2,
        window['start'],
        window['end'],
        token_list,
    )

    example = PaReco(data)
    example.verboseMode(False)
    interval_runs[data_number] = example

    try:
        # Defense-in-depth: never execute a variant check beyond the global hard limit,
        # even if an interval is later edited or supplied manually.
        if as_datetime(window['end']) > HARD_VARIANT_END:
            raise ValueError(
                f"Window end {window['end']} exceeds the hard variant limit "
                f"{HARD_VARIANT_END_TIMESTAMP}."
            )

        prs_source = example.extractPatches(window['start'], window['end'])
        pr_count = len(prs_source)

        if pr_count > 0:
            example.runClassification(prs_source)
            status = 'Completed'
        else:
            status = 'No PRs found'

    except Exception as exc:
        pr_count = None
        status = f'Failed: {type(exc).__name__}: {exc}'
        print(status)

    run_summary.append({
        'data_number': data_number,
        'variant_index': variant_index,
        'variant1': variant1,
        'variant2': variant2,
        'label': window['label'],
        'start': window['start'],
        'end': window['end'],
        'pr_count': pr_count,
        'status': status,
    })

run_summary_columns = [
    'data_number',
    'variant_index',
    'variant1',
    'variant2',
    'label',
    'start',
    'end',
    'pr_count',
    'status',
]
run_summary_df = pd.DataFrame(run_summary, columns=run_summary_columns)


Display the status and PR count for every interval across all configured repository pairs.


In [ ]:
run_summary_df


Inspect one completed interval by selecting its globally unique data ID. Use `interval_plan` or `run_summary_df` to find the data ID belonging to a particular pair and date range.


In [ ]:
selected_data_number = '430'
selected_example = interval_runs.get(selected_data_number)
selected_run = run_summary_df[
    run_summary_df['data_number'] == selected_data_number
]

if not selected_run.empty:
    display(selected_run)

if selected_example is None:
    print(f'No run exists for data number {selected_data_number}.')
elif hasattr(selected_example, 'df_patches'):
    display(selected_example.dfPatches())
else:
    print(f'Data number {selected_data_number} did not finish patch extraction.')


Inspect patch-level classifications for the selected interval.


In [ ]:
if selected_example is not None and hasattr(selected_example, 'df_patch_classes'):
    display(selected_example.dfPatchClass())
else:
    print(f'Data number {selected_data_number} has no patch classification dataframe.')


In [ ]:
# Inspect file-level classifications for the selected interval.
if selected_example is not None and hasattr(selected_example, 'df_files_classes'):
    display(selected_example.dfFileClass())
else:
    print(f'Data number {selected_data_number} has no file classification dataframe.')
